In [5]:
import torch.optim as optim
import matplotlib.pyplot as plt

from src.load_and_save import save_model
from src.model import SimpleNN
from src.pruning import get_intermediate_outputs_as_numpy
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

from src.pruning import is_all_layers_separated
import numpy as np



In [6]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [7]:
from src.improved_model import SimpleCNN

# Instantiate the model
model = SimpleCNN(0.0).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=float(1e-2))

In [ ]:
from src.training import get_average_separation

# Training loop
num_epochs: int = 200
target_accuracy: float = .95
maximum_scramble_distance: float = 5.0

test_data, _ = next(iter(test_dataloader))
test_data.to(device)


for epoch in range(num_epochs):
    for inputs, labels in train_dataloader:
        if epoch == 0:
            break
        # Forward pass
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # separation: float = get_average_separation(test_data, model)

    # Assess progress:
    # data: np.ndarray = get_intermediate_outputs_as_numpy(model, train_dataloader)
    validation_accuracy: float = get_accuracy(model, val_dataloader)
    if validation_accuracy > target_accuracy:
        model.scramble_distance = min(model.scramble_distance + .25, maximum_scramble_distance)
        print(model.scramble_distance)
    print(f'Epoch [{epoch+1}/{num_epochs}], Accuracy: {get_accuracy(model, val_dataloader):.4f}')


Epoch [1/200], Accuracy: 0.0896
Epoch [2/200], Accuracy: 0.8982
Epoch [3/200], Accuracy: 0.9277
Epoch [4/200], Accuracy: 0.9347
Epoch [5/200], Accuracy: 0.9423
Epoch [6/200], Accuracy: 0.9457
0.25
Epoch [7/200], Accuracy: 0.9523
0.5
Epoch [8/200], Accuracy: 0.9541
0.75
Epoch [9/200], Accuracy: 0.9534
1.0
Epoch [10/200], Accuracy: 0.9532
Epoch [11/200], Accuracy: 0.9442
Epoch [12/200], Accuracy: 0.9488
1.25
Epoch [13/200], Accuracy: 0.9517
1.5
Epoch [14/200], Accuracy: 0.9501
Epoch [15/200], Accuracy: 0.9456
Epoch [16/200], Accuracy: 0.9496
Epoch [17/200], Accuracy: 0.9469
1.75
Epoch [18/200], Accuracy: 0.9518
Epoch [19/200], Accuracy: 0.9500
2.0
Epoch [20/200], Accuracy: 0.9502
Epoch [21/200], Accuracy: 0.9475
Epoch [22/200], Accuracy: 0.9488
Epoch [23/200], Accuracy: 0.9467
2.25
Epoch [24/200], Accuracy: 0.9509
2.5
Epoch [25/200], Accuracy: 0.9520
Epoch [26/200], Accuracy: 0.9482
2.75
Epoch [27/200], Accuracy: 0.9520
Epoch [28/200], Accuracy: 0.9488
Epoch [29/200], Accuracy: 0.9477
Ep

In [ ]:
from src.model import scramble_activation

test_data = test_data.to(device)
model.eval_mode()
model(test_data[0:1])



scramble_activation(model.layer1(test_data[0:1]), model.scramble, model.scramble_distance)
# model(test_data[0,...])
